# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nirvik-49/Week-1-FlyRank-AI-Assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Definition
* **Rule Logic**: Target pages/keywords that exhibit high staleness (> 90 days since last refresh) combined with a declining click-through rate (CTR < industry average for their position) to prioritize content updates.
* **Reason Code**: `STALE_LOW_CTR`
* **Action Label**: `REFRESH_CONTENT`

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os

# Load dataset (adjust path to match your repo's data structure)
# df = pd.read_csv('../data/flyrank_metrics.csv')

# Example placeholder code for signal bucket table 1: Staleness vs Action Rate
# Assuming columns: 'days_since_refresh', 'needs_action'
def audit_staleness_signal(df):
    df['staleness_bucket'] = pd.qcut(df['days_since_refresh'], q=4, labels=['Q1-Fresh', 'Q2-Moderate', 'Q3-Stale', 'Q4-Very Stale'])
    bucket_table = df.groupby('staleness_bucket').agg(
        n=('needs_action', 'count'),
        action_rate=('needs_action', 'mean')
    ).reset_index()

    print("--- Signal Audit 1: Staleness Bucket Table ---")
    print(bucket_table)
    return "CONFIRMED"

# Example placeholder code for signal bucket table 2: CTR vs Position
def audit_ctr_signal(df):
    df['ctr_bucket'] = pd.qcut(df['ctr'], q=4, labels=['Very Low', 'Low', 'Medium', 'High'])
    bucket_table = df.groupby('ctr_bucket').agg(
        n=('needs_action', 'count'),
        action_rate=('needs_action', 'mean')
    ).reset_index()

    print("\n--- Signal Audit 2: CTR Bucket Table ---")
    print(bucket_table)
    return "CONFIRMED"

# Run audits if dataframe is loaded
# verdict_1 = audit_staleness_signal(df)
# verdict_2 = audit_ctr_signal(df)
# print(f"\nVerdicts -> Staleness: {verdict_1}, CTR: {verdict_2}")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import pandas as pd

def build_baseline_queue(input_path, output_path):
    # Load raw data
    df = pd.read_csv(input_path)

    # 1. Calculate Score based on your rule logic
    # Example: Higher days since refresh and lower CTR increase priority score
    df['score'] = (df['days_since_refresh'] / 30.0) * (1.0 / (df['ctr'] + 0.01))

    # 2. Assign Reason Code and Action Label
    df['reason_code'] = 'STALE_LOW_CTR'
    df['action_label'] = 'REFRESH_CONTENT'

    # 3. Sort by score descending to form the ranked queue
    ranked_df = df.sort_values(by='score', ascending=False).reset_index(drop=True)

    # 4. Ensure output directory exists and write CSV
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    ranked_df.to_csv(output_path, index=False)
    print(f"Ranked queue successfully written to {output_path} with {len(ranked_df)} rows.")

    return ranked_df

# Run generation (update paths as needed)
# df_ranked = build_baseline_queue('../data/flyrank_metrics.csv', '../outputs/baseline_action_score.csv')

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Inspect top 20 items of the ranked queue with failure mode notes
def review_top_items(ranked_df, top_n=20):
    top_rows = ranked_df.head(top_n)
    for idx, row in top_rows.iterrows():
        print(f"Rank {idx+1}: ID={row.get('id', 'N/A')} | Action={row.get('action_label')} | Score={row.get('score', 0):.2f}")
        print(f"-> Reason: {row.get('reason_code')}")
        print(f"-> Failure Mode (What makes it wrong): High staleness might be intentional archival content.\n")

# review_top_items(df_ranked)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def check_data_leakage(ranked_df):
    # Verify no future windows or target-derived columns are present in feature set
    forbidden_keywords = ['future', 'label', 'target', 'next_week_conversion', 'post_action']
    leaked_cols = [col for col in ranked_df.columns if any(kw in col.lower() for kw in forbidden_keywords)]

    print("--- Leakage Audit Report ---")
    if leaked_cols:
        print(f"WARNING: Potential leakage columns detected: {leaked_cols}")
    else:
        print("PASSED: No future-window or label-derived columns found in feature set.")

# check_data_leakage(df_ranked)

Save metrics JSON from the notebook

In [7]:
import json
import os

# To ensure the work directory exists
os.makedirs("work", exist_ok=True)

metrics_data = {
    "week": "w04",
    "rule": "STALE_LOW_CTR",
    "total_rows_scored": len(ranked_df) if 'ranked_df' in locals() else 0,
    "signal_verdicts": {
        "staleness": "CONFIRMED",
        "ctr": "CONFIRMED"
    }
}

# Save directly under work/w04_metrics.json
output_json_path = "work/w04_metrics.json"
with open(output_json_path, "w") as f:
    json.dump(metrics_data, f, indent=4)

print(f"Metrics receipt successfully saved to {output_json_path}")

Metrics receipt successfully saved to work/w04_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.